# Fetch the data

In [0]:
import requests
import pandas as pd
from datetime import datetime, timedelta, timezone

end = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
start = end - timedelta(days=7)


from_str = start.strftime("%Y-%m-%dT%H:%MZ")
to_str = end.strftime("%Y-%m-%dT%H:%MZ")
                                                                                                                                                            
url = f"https://api.carbonintensity.org.uk/regional/intensity/{from_str}/{to_str}"
response = requests.get(url, timeout=30)
response.raise_for_status()
data = response.json()

print(f"Status: {response.status_code}")
print(f"Top-level keys: {list(data.keys())}")
print(f"First period keys: {list(data['data'][0].keys())}")
print(f"First region keys: {list(data['data'][0]['regions'][0].keys())}")



In [0]:
print(f"Intensity structure: {data['data'][0]['regions'][0]['intensity']}")

In [0]:
import sys
sys.path.insert(0, "/Workspace/Users/mrjameswestwood@gmail.com/EV-Charging-Demand-Optimisation/energy-forecasting")
from src.data.collectors.carbon_intensity import fetch_regional_carbon_intensity                                                                            
from datetime import datetime, timedelta, timezone
                                                                                                                                                            
end = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)                                                                                 
start = end - timedelta(days=7)                                                                                                                             
                                                                                                                                                
# Note: fetches via pandas on the driver — acceptable for this data volume.
# At scale, replace with a scheduled Databricks job writing directly to Delta                                     
# or use Autoloader for continuous ingestion.

df = fetch_regional_carbon_intensity(start, end)
display(df)

### Write a Delta Table for the Bronze layer

In [0]:
sdf = spark.createDataFrame(df)
# Bronze layer delta table
sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_carbon_intensity_regional")